In [2]:
import pickle
from sklearn.model_selection import train_test_split
all_data = pickle.load(open("para_data.pkl", "rb"))

features = ["adj_noun_ratio", "avg_depth", "fk_grade", "ttr", "hapax", ";", "!", "—"]

import numpy as np

data_X = np.zeros((len(all_data), len(features)))
data_y = [0] * len(all_data)

for j, para in enumerate(all_data):
    for i, feature in enumerate(features):
        if feature in [";", "!", "—"]:
            data_X[j, i] = para['punctuation'].get(feature, 0)
        else:
            data_X[j, i] = para[feature]
    data_y[j] = 0 if para["label"] == "human" else 1
# print(data_y)

X_train, X_test, y_train, y_test = train_test_split(data_X, data_y, test_size=0.2, random_state=67)

Tier A, XGBoost model

In [17]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

estimator = XGBClassifier(eval_metric='logloss')

testing_grid = {
    'n_estimators': [302, 303, 304],
    'max_depth': [1, 2, 3],
    'learning_rate': [0.09, 0.1, 0.11],
}

search = GridSearchCV(estimator, testing_grid, scoring='roc_auc', cv=5, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)

print("Best parameters found: ", search.best_params_)
print("Best score: ", search.best_score_)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\nTest set performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters found:  {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 303}
Best score:  0.9880249828949766

Test set performance:
Accuracy: 0.9447004608294931
Precision: 0.9294871794871795
Recall: 0.9539473684210527
F1: 0.9415584415584416
ROC AUC: 0.983163961777643
Confusion matrix:
 [[325  22]
 [ 14 290]]


In [3]:
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

embeddings = {}
with open("glove/glove.6B.300d.txt", encoding="utf8") as f:
    for line in f:
        values = line.strip().split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings[word] = vector


X_data = []
stop_words = set(stopwords.words('english'))

for para in all_data:
    paragraph = para['text'].lower()
    tokens = word_tokenize(paragraph)
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    # tokens = [word for word in tokens if word.isalpha()]
    vectors = [embeddings[word] for word in tokens if word in embeddings]
    if vectors:
        X_data.append(np.mean(vectors, axis=0))
    else:
        X_data.append(np.zeros(300))


X_data = np.array(X_data)
y_data = np.array([0 if para["label"] == "human" else 1 for para in all_data])

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=69)


In [4]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch

class predictor(nn.Module):
    def __init__(self):
        super(predictor, self).__init__()
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

        self.fc1 = nn.Linear(300, 128)
        self.fc2 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 2)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.relu(self.fc2(x))
        x = self.dropout(x)

        x = self.out(x)
        return x
    

train_data = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long())
test_data = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test).long())

batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size)

model = predictor()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())

print("\nTest set performance:")
print("Accuracy:", accuracy_score(y_test, all_preds))
print("Precision:", precision_score(y_test, all_preds))
print("Recall:", recall_score(y_test, all_preds))
print("F1:", f1_score(y_test, all_preds))
print("Confusion matrix:\n", confusion_matrix(y_test, all_preds))


Epoch 1/10, Loss: 0.2242
Epoch 2/10, Loss: 0.0296
Epoch 3/10, Loss: 0.0224
Epoch 4/10, Loss: 0.0183
Epoch 5/10, Loss: 0.0135
Epoch 6/10, Loss: 0.0122
Epoch 7/10, Loss: 0.0086
Epoch 8/10, Loss: 0.0042
Epoch 9/10, Loss: 0.0029
Epoch 10/10, Loss: 0.0024

Test set performance:
Accuracy: 0.9877112135176651
Precision: 0.9771986970684039
Recall: 0.9966777408637874
F1: 0.9868421052631579
Confusion matrix:
 [[343   7]
 [  1 300]]
